# 3. Control con Aproximaciones: Métodos Lineales

Habiendo comprobado que los métodos puramente tabulares sufren la 'maldición de la dimensionalidad' en entornos continuos, en este cuaderno abordamos la **aproximación de funciones**, donde sustituimos la matriz $Q(s, a)$ por una función matemática parametrizada $\hat{q}(s,a,\mathbf{w})$.

Implementaremos el método **Semi-Gradiente Episódico** como primer paso antes del Deep Learning, usando una combinación lineal básica de características matemáticas del estado continuo.

In [ ]:
import gymnasium as gym
import numpy as np
from sklearn.kernel_approximation import RBFSampler

# Importar los componentes modulares
from src.agents.agent import Agent
from src.policies.epsilon_greedy import EpsilonGreedyPolicy
from src.learners.sarsa_semi_gradient import SARSASemiGradient
from src.plotting.plotting import plot_rewards, plot_episode_lengths

## Extracción de Características (Feature Engineering)
Como el vector de MountainCar sólo tiene 2 dimensiones `[pos, vel]`, aplicar pesos lineales puros a esas 2 variables sería insuficiente para captar la complejidad de subir la colina ($W \cdot X$ aprendería una recta). 

Por ello, usamos una aproximación basada en funciones de base radial (RBF), que expanden esas 2 variables a un espacio dimensional mucho más alto (100 dimensiones) permitiendo que la combinación lineal sea altísimamente no lineal y expresiva.

In [ ]:
env_name = "MountainCar-v0"
env = gym.make(env_name)

# La herramienta RBF generará las características complejas basadas en Gaussianas
# Es una técnica estándar de Machine Learning Clásico para aproximar funciones no lineales
rbf_feature = RBFSampler(gamma=1.0, n_components=100, random_state=42)
# Ajustamos (fijamos) el RBF sobre unos ejemplos inventados que cubran todo el espacio:
state_samples = np.array([env.observation_space.sample() for _ in range(10000)])
rbf_feature.fit(state_samples)

def rbf_extractor(state):
    """
    Recibe el estado continuo (2,) y devuelve un vector de características (100,).
    """
    return rbf_feature.transform([state])[0]

feature_size = 100
action_size = env.action_space.n

## Entrenamiento del Agente Lineal

Ya no instanciamos `QLearning` o `SARSA` tabular, sino que enviamos nuestro modelo de pesos al entorno.

In [ ]:
n_episodes = 500 # Debería ser mucho más rápido que la versión tabular de la Fase 3
n_runs = 5
gamma = 0.99
alpha = 0.1 # Tasa de aprendizaje para los pesos

policy = EpsilonGreedyPolicy(epsilon_start=1.0, epsilon_min=0.01, epsilon_decay=0.99)
semi_grad_learner = SARSASemiGradient(action_size=action_size, 
                                      feature_size=feature_size, 
                                      alpha=alpha, 
                                      gamma=gamma, 
                                      feature_extractor=rbf_extractor)

print("Entrenando Semi-Gradiente Lineal (RBF) en MountainCar...")
agent = Agent(env, semi_grad_learner, policy)
weights, rewards, lengths, stats = agent.train(num_episodes=n_episodes, n_runs=n_runs)

## Visualización
Compárese esto con los resultados de la discretización: el tiempo de convergencia es infinitamente más rápido a pesar de lidiar con números puramente continuos.

In [ ]:
plot_rewards([rewards], legend_labels=["Semi-Gradiente Lineal RBF"], log_scale=False)
plot_episode_lengths([lengths], legend_labels=["Semi-Gradiente Lineal RBF"], log_scale=False)